In [ ]:
import tkinter as tk
from tkinter import scrolledtext, Entry, Button, Frame
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


MODEL_PATH = r"\Desktop\y_final_genius_bot_v5"


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)
model.eval()

def get_bot_response(user_message):
    prompt = f"Вы: {user_message}\nБот:" 

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=1.1,
        top_k=178,
        top_p=0.95,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=final_tokenizer.eos_token_id,
        no_repeat_ngram_size=3
    )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Убираем prompt из результата (если повторился)
    response_only = full_response.replace(prompt, "").strip()

    cleaned_response = clean_response(response_only)
    return cleaned_response


def clean_response(text):
    text = re.split(r'#|###|Собеседник:|Ты:', text)[0]
    # Убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    # Если ответ пустой, возвращаем заглушку
    if not text:
        return "Я не знаю, что на это ответить."
    return text


class ChatApplication(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Чат с ботом")
        self.geometry("800x600") 
        self.configure(bg="#2c3e50") 
        self._create_widgets()

    def _create_widgets(self):
        chat_font = ("Arial", 12)
        self.chat_area = scrolledtext.ScrolledText(
            self,
            wrap=tk.WORD, 
            state='disabled', 
            font=chat_font,
            bg="#34495e", 
            fg="#ecf0f1"  
        )
        self.chat_area.pack(padx=10, pady=10, fill=tk.BOTH, expand=True)


        self.chat_area.tag_config('user', foreground="#3498db", font=(chat_font[0], chat_font[1], "bold")) # Синий для пользователя
        self.chat_area.tag_config('bot', foreground="#2ecc71") 

        input_frame = Frame(self, bg="#2c3e50")
        input_frame.pack(padx=10, pady=(0, 10), fill=tk.X)

        self.input_entry = Entry(
            input_frame, 
            font=chat_font,
            bg="#34495e",
            fg="#ecf0f1",
            insertbackground="#ecf0f1" 
        )
        self.input_entry.pack(side=tk.LEFT, fill=tk.X, expand=True, ipady=5)
        self.input_entry.bind("<Return>", self.send_message)


        self.send_button = Button(
            input_frame,
            text="Отправить",
            command=self.send_message,
            font=("Arial", 11, "bold"),
            bg="#2980b9",
            fg="#ffffff",
            activebackground="#3498db",
            activeforeground="#ffffff",
            relief=tk.FLAT, 
            padx=10
        )
        self.send_button.pack(side=tk.RIGHT, padx=(10, 0))

    def send_message(self, event=None):
        user_input = self.input_entry.get()
        if not user_input.strip():
            return 

        self._display_message("Вы: ", user_input, "user")


        bot_response = get_bot_response(user_input)
        self._display_message("Бот: ", bot_response, "bot")
        
        self.input_entry.delete(0, tk.END)

    def _display_message(self, sender, message, tag):
        self.chat_area.config(state='normal') 
        self.chat_area.insert(tk.END, sender, tag)
        self.chat_area.insert(tk.END, message + "\n\n")

        self.chat_area.config(state='disabled') 
        self.chat_area.see(tk.END) 
